# nanoCalc-GPT Evaluation
Evaluate the saved model checkpoint on train and test datasets using exact match accuracy and per-operator breakdowns.

In [1]:
from exact_match import load_saved_model, exact_match, exact_match_by_operator, operator_accuracy, predict

# Load saved model from artifacts
model, config = load_saved_model("artifacts/model.pt")
print(f"Loaded model successfully on device: {config.device}")

Loaded model successfully on device: cpu


In [2]:
# Evaluate exact match accuracy on Test set with formatted operator table
test_results = exact_match(model, data="test")
print(test_results.summary(show_table=True))

=== Exact Match | Split: test ===
Overall Accuracy: 2794/3057 (91.3968%)

+--------------------+---------+---------+--------+----------+
| Operator           |   Total | Correct | Errors | Accuracy |
+--------------------+---------+---------+--------+----------+
| * (Multiplication) |    1000 |     749 |    251 |  74.90% |
| + (Addition)       |    1000 |    1000 |      0 | 100.00% |
| - (Subtraction)    |    1000 |     999 |      1 |  99.90% |
| / (Division)       |      57 |      46 |     11 |  80.70% |
+--------------------+---------+---------+--------+----------+
| Overall            |    3057 |    2794 |    263 |  91.40% |
+--------------------+---------+---------+--------+----------+


In [3]:
# Per-operator accuracy queries and metrics
for op in test_results.operators:
    name = test_results.by_operator[op]['name']
    acc = test_results.op_accuracy(op)
    corr = test_results.op_correct(op)
    tot = test_results.op_total(op)
    errs = test_results.op_errors(op)
    print(f"{op} ({name}): {corr}/{tot} = {acc:.2%} (Errors: {errs})")

# Inspect error cases for multiplication
print("\n--- Multiplication Error Samples ---")
for s in test_results.get_errors('*', limit=3):
    print(f"{s['prompt']} -> Pred: {s['predicted']} (decoded: {s['decoded_pred']}) | Expected: {s['target']} (expected: {s['decoded_target']})")

* (Multiplication): 749/1000 = 74.90% (Errors: 251)
+ (Addition): 1000/1000 = 100.00% (Errors: 0)
- (Subtraction): 999/1000 = 99.90% (Errors: 1)
/ (Division): 46/57 = 80.70% (Errors: 11)

--- Multiplication Error Samples ---
96*91= -> Pred: +6188 (decoded: 8816) | Expected: +6378 (expected: 8736)
84*48= -> Pred: +2704 (decoded: 4072) | Expected: +2304 (expected: 4032)
47*82= -> Pred: +4383 (decoded: 3834) | Expected: +4583 (expected: 3854)


In [4]:
# Filter evaluation to a single operator directly
mult_only = exact_match(model, data="test", operator="*")
print(mult_only.summary(show_table=False))

=== Exact Match | Split: test | Filter: * ===
Overall Accuracy: 749/1000 (74.9000%)
Per-Operator Accuracy:
  Op '*' (Multiplication): 749/1000 (74.9000%)


In [5]:
# Evaluate exact match accuracy on Train set
train_results = exact_match(model, data="train")
print(train_results.summary(show_table=True))

=== Exact Match | Split: train ===
Overall Accuracy: 36000/36000 (100.0000%)

+--------------------+---------+---------+--------+----------+
| Operator           |   Total | Correct | Errors | Accuracy |
+--------------------+---------+---------+--------+----------+
| * (Multiplication) |    9000 |    9000 |      0 | 100.00% |
| + (Addition)       |    9000 |    9000 |      0 | 100.00% |
| - (Subtraction)    |    9000 |    9000 |      0 | 100.00% |
| / (Division)       |    9000 |    9000 |      0 | 100.00% |
+--------------------+---------+---------+--------+----------+
| Overall            |   36000 |   36000 |      0 | 100.00% |
+--------------------+---------+---------+--------+----------+


In [6]:
# Interactive predictions
prompts = ["12+34=", "85-23=", "15*06=", "72/08=", "99*99="]
for p in prompts:
    res = predict(model, p)
    print(f"{res['prompt']} -> Raw: {res['raw_prediction']} | Decoded: {res['math_value']}")

12+34= -> Raw: +6400 | Decoded: 46
85-23= -> Raw: +2600 | Decoded: 62
15*06= -> Raw: +0900 | Decoded: 90
72/08= -> Raw: +9000 | Decoded: 9
99*99= -> Raw: +1089 | Decoded: 9801
